In [1]:
import sys
sys.path.append('../')
import UTILS.utils as utils

import pandas as pd
import string
import google.generativeai as genai


### SPLIT INTO CHAPTERS

In [2]:
# test if any line is a chapter title
def is_chapter_title(current_chapter, line):
    if(current_chapter < 9):
        if line[:1].isdigit() and line[1] == "." :
            if int(line[:1]) == current_chapter + 1:
                return True
    elif (current_chapter < 99) and len(line) >= 3:
         if line[:2].isdigit() and line[2] == "." :
            if int(line[:2]) == current_chapter + 1:
                return True
    elif (current_chapter == 99) and len(line) >= 4:
        if line[:3].isdigit() and line[3] == "." and int(line[:3]) == 100 :
            return True

In [3]:
# detect chapter title in page
def detect_chapter_title(current_chapter_num, page_lines):
    for i, line in enumerate(page_lines):
        if(is_chapter_title(current_chapter_num, line)):
            return i
    return -1

In [4]:
all_files = utils.get_files_in_directory(utils.ALL_PAGES_TXT_VISION_PARAGRAPHS)

In [5]:
# split full text into its chapters
def split_into_chapters():
    chapter_num = 0
    chapters = []
    current_chapter = []
    for file in all_files:
        page_paragraphs = utils.read_file(file).split("\n")

        page_paragraphs = [line for line in page_paragraphs if line.upper() != utils.BOOK_TITLE]
        page_paragraphs = [line for line in page_paragraphs if not line.replace("°", "0").isdigit()]

        new_chapter_line = detect_chapter_title(chapter_num, page_paragraphs)
        if new_chapter_line == -1:
            current_chapter += page_paragraphs
        else:
            current_chapter += page_paragraphs[:new_chapter_line]
            chapters.append(current_chapter)
            current_chapter = page_paragraphs[new_chapter_line:]
            chapter_num += 1

    
    chapters.append(current_chapter)
    return chapters

### PARSE CHAPTERS

In [6]:
# breaks down chapter into paragraphs (corrects parsing errors from Vision)
def parse_chapter(chapter):
    parsed_chapter = [] # tracks chapter
    current_paragraph = "" # tracks current paragraph
    for line in chapter:
        if len(line) > 2: # if line is longer than 2 characters

            if line[-1] not in string.punctuation:
                # if line doesn't end in punctuation, add line and space
                current_paragraph += line + " "

            elif line[-1] == "-" :
                # if line ends in hyphen, remove hyphen
                current_paragraph += line[:-1]

            elif line[:-1].isdigit() and line[-1] == ".":
                # special case if chapter title was parsed as 2 lines
                current_paragraph += line + " "

            else:
                # otherwise if line ends in punctuation, symbolizes end of paragraph 
                current_paragraph += line
                parsed_chapter.append(current_paragraph)
                current_paragraph = ""

    return parsed_chapter

In [7]:
def parse_all_chapters():
    for i, chapter in enumerate(split_into_chapters()):
        parsed_chapter = parse_chapter(chapter)
        file_name = utils.ALL_CHAPTERS + "chapter_" + str(i) + ".txt"
        utils.write_to_file(file_name, "\n".join(parsed_chapter), True)

#parse_all_chapters()

In [8]:
all_chapters = []
for chapter in utils.get_files_in_directory(utils.ALL_CHAPTERS):
    all_chapters.append(utils.read_file(chapter))
all_chapters = all_chapters[1:]

all_sentences = []
all_paragraphs = []
for chapter in all_chapters:
    for paragraph in chapter.split("\n"):
        all_paragraphs.append(paragraph)
        for sentence in paragraph.split("."):
            all_sentences.append(sentence)

### PARSE SENTENCES

In [15]:
genai.configure(api_key=utils.API_KEY())
model = genai.GenerativeModel('gemini-1.5-flash')

prompt = """Your task is to extract all sentences from the following paragraph of text and return each sentence on an individual line. 
For example, if the paragraph is "Today is nice. Tomorrow is better." you should return "Today is nice.\n Tomorrow is better.\n"
The paragraph is :
"""

def prompt_model(prompt):
    response = model.generate_content(contents=[prompt])
    return response.text

In [10]:
# fixes various parsing errors
def correctly_parse_chapter(chapter):
    paragraphs = chapter.split("\n")
    paragraphs_new = []
    current_paragraph = ""
    # for each paragraph
    for i, paragraph in enumerate(paragraphs):
        if i == (len(paragraphs) - 1):
            # if final paragraph in chapter
            paragraphs_new.append((current_paragraph + " " + paragraph).strip())
        else:
            split_paragraph = paragraph.split(" ")

            # if paragraph has page number in it (parsing error)
            if(split_paragraph[0].isdigit()):
                current_paragraph += " " + " ".join(split_paragraph[1:])
            else: # otherwise do nothing
                current_paragraph += " " + paragraph

            # if next paragraph does not start with capital letter, there is an error in parsing
            # we continue keeping the current paragraph
            if (not paragraphs[i + 1][0].isupper()):
                continue
            else:
                paragraphs_new.append(current_paragraph.strip())
                current_paragraph = ""
    return paragraphs_new

In [11]:
all_chapters_parsed = [correctly_parse_chapter(chapter) for chapter in all_chapters]

In [17]:
from tqdm import tqdm

poem_chapters = [9, 29, 92] # special poem chapters


def get_sentences_from_gemini(chapters):
    sentences_per_chapter = []
    for i, chapter in tqdm(enumerate(chapters)): # for each chapter

        # create directory
        chapter_directory = "CHAPTER_" + str(i + 1)
        utils.create_directory(utils.ALL_SENTENCES + chapter_directory)

        # if poem chapter, parse entire chapter
        if((i + 1) in poem_chapters):
            current_prompt = prompt + " ".join(chapter)
            response = prompt_model(current_prompt)
            sentences_per_chapter.append((i + 1, response))
            file_name = utils.ALL_SENTENCES + chapter_directory + "/paragraph_1.txt"
            utils.write_to_file(file_name, response, True)

        # othrwise, parse chapter paragraph by paragraph    
        else:
            for j, paragraph in enumerate(chapter):
                
                current_prompt = prompt + paragraph
                response = prompt_model(current_prompt)

                sentences_per_chapter.append((i + 1, response))

                file_name = utils.ALL_SENTENCES + chapter_directory + "/paragraph_" + str(j + 1) + ".txt"
                utils.write_to_file(file_name, response, True)

    return sentences_per_chapter

#sentences_per_chapter = get_sentences_from_gemini(all_chapters_parsed)

### CREATE SENTENCES DATAFRAME

In [2]:
## create sentence dataframe from individual files

def get_chapter(file_path):
    return file_path.split("/")[-2].split("_")[-1]

def get_all_sentences_df():
    all_files = []
    all_folders = utils.get_folders_in_folder(utils.ALL_SENTENCES)
    for folder in all_folders:
        all_files += utils.get_files_in_directory(folder)

    all_sentences = []
    for file in all_files:
        text_in_file = utils.read_file(file)
        sentences = text_in_file.split("\n")
        chapter_num = get_chapter(file)
        for sentence in sentences:
            all_sentences.append((sentence, chapter_num))
    return pd.DataFrame(all_sentences, columns=["text", "chapter"])

all_sentences = get_all_sentences_df()

In [30]:
all_sentences["length"] = all_sentences["text"].apply(lambda x : len(x)) 
all_sentences[all_sentences["length"] > 0] # remove empty lines

,text,chapter,length,tokens,num_tokens
0,1. Britain's Rich Mart of the Orient-Hongkong ...,1,62,"[Britain, 's, Rich, Mart, Orient-Hongkong, Har...",6
2,We are on the upper deck of one of the many st...,1,291,"[upper, deck, one, many, steamers, ride, ancho...",25
3,"We are not, however, yet in China.",1,34,"[however, yet, China]",3
4,We are looking southwest and the mainland lies...,1,163,"[looking, southwest, mainland, lies, right, di...",15
5,A little to the left of the highest point of t...,1,172,"[little, left, highest, point, somber, elevati...",17
...,...,...,...,...,...
4227,We have been stoned by the superstitious rusti...,100,102,"[stoned, superstitious, rustics, among, mounta...",9
4228,We have looked upon the bloody and harrowing c...,100,223,"[looked, upon, bloody, harrowing, circumstance...",15
4229,She is weak by reason of her unpreparedness fo...,100,118,"[weak, reason, unpreparedness, defense, vultur...",10
4230,Even now she has ceased to be a sovereign powe...,100,193,"[Even, ceased, sovereign, power, allied, natio...",15


In [31]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# tokenize sentences and remove those with too few tokens
MIN_TOKENS = 10

stop_words = set(stopwords.words('english'))

def filter_sentence(sentence):
    word_tokens = word_tokenize(sentence)
    tokens = [w for w in word_tokens if not w.lower() in stop_words and len(w) > 1]
    return tokens

df = all_sentences
df["tokens"] = df["text"].apply(lambda x : filter_sentence(x))
df["num_tokens"] = df["tokens"].apply(lambda x : len(x))
df = df[df["num_tokens"] >= MIN_TOKENS]
df = df.reset_index()


In [32]:
df_to_save = df.drop(["index", "tokens", "num_tokens"], axis=1)
df_to_save.to_csv(utils.ALL_SENTENCES_DF, index=False)

### CREATE PARAGRAPHS DATAFRAME

In [ ]:
all_sentences = utils.get_folders_in_folder(utils.ALL_SENTENCES)

In [ ]:
all_paragraphs = []
for i, chapter in enumerate(all_sentences):
    all_files = utils.get_files_in_directory(chapter + "/")
    for j, file in enumerate(all_files):
        all_paragraphs.append({"chapter" : i+1, "paragraph" : j+1, "text" : utils.read_file(file)})

all_paragraphs_df = pd.DataFrame.from_dict(all_paragraphs)
all_paragraphs_df_filtered = all_paragraphs_df[all_paragraphs_df["paragraph"] > 1]

In [10]:
all_paragraphs_df_filtered.to_csv("../../Data/TEXT/ALL_PARAGRAPHS.csv", index=False)